# **Part 1 – Data Loading & Exploration**


- Loaded the training, testing, and ground truth datasets.
- Explored dataset structure, data types, and statistical summaries.
- Checked data quality and feature distributions

Import Libraries

In [1]:
# --- Project setup: make src/ importable and load the configuration ---
import json
import os
import sys

REPO_ROOT = os.path.abspath("..") if os.path.isdir("../src") else os.path.abspath(".")
if not os.path.isdir(os.path.join(REPO_ROOT, "src")):
    # Google Colab: clone the repository so src/ and configs/ are available.
    !git clone https://github.com/yomna-0/nasa-predictive-maintenance-rul.git
    os.chdir("nasa-predictive-maintenance-rul/notebooks")
    REPO_ROOT = os.path.abspath("..")

sys.path.insert(0, REPO_ROOT)

with open(os.path.join(REPO_ROOT, "configs", "default.json")) as f:
    CONFIG = json.load(f)

SEED = CONFIG["seed"]
VAL_SIZE = CONFIG["val_size"]
WINDOW_SIZE = CONFIG["window_size"]

import pandas as pd
import numpy as np
from IPython.display import display

from src.data import (
    add_rul,
    split_by_engine,
    find_constant_features,
    fit_scaler,
    create_sequences,
    create_test_sequences,
    save_prepared_data,
    save_scaler,
)

In [2]:
# On Google Colab: upload PM_train.csv, PM_test.csv and PM_truth.csv when asked.
# Locally: keep the three files in data/ (repo root) or next to this notebook; nothing is asked.
DATA_DIR = "."
for candidate in ["data", "../data", "."]:
    if os.path.exists(os.path.join(candidate, "PM_train.csv")):
        DATA_DIR = candidate
        break

if not os.path.exists(os.path.join(DATA_DIR, "PM_train.csv")):
    from google.colab import files
    files.upload()

Load Dataset

In [3]:
train_df = pd.read_csv(os.path.join(DATA_DIR, "PM_train.csv"))
test_df = pd.read_csv(os.path.join(DATA_DIR, "PM_test.csv"))
truth_df = pd.read_csv(os.path.join(DATA_DIR, "PM_truth.csv"))

print("Datasets loaded successfully!")

print("Training Data:", train_df.shape)
print("Testing Data:", test_df.shape)
print("Truth Data:", truth_df.shape)

Datasets loaded successfully!
Training Data: (20631, 26)
Testing Data: (13096, 26)
Truth Data: (100, 2)


Preview the Data

In [4]:
def preview_data(df, name):
    print(f"\n{name} Dataset")
    print("-" * 40)
    display(df.head())

preview_data(train_df, "Training")
preview_data(test_df, "Testing")
preview_data(truth_df, "Truth")


Training Dataset
----------------------------------------


,id,cycle,setting1,setting2,setting3,s1,s2,s3,s4,s5,...,s12,s13,s14,s15,s16,s17,s18,s19,s20,s21
0,1,1,-0.0007,-0.0004,100.0,518.67,641.82,1589.70,1400.60,14.62,...,521.66,2388.02,8138.62,8.4195,0.03,392,2388,100.0,39.06,23.4190
1,1,2,0.0019,-0.0003,100.0,518.67,642.15,1591.82,1403.14,14.62,...,522.28,2388.07,8131.49,8.4318,0.03,392,2388,100.0,39.00,23.4236
2,1,3,-0.0043,0.0003,100.0,518.67,642.35,1587.99,1404.20,14.62,...,522.42,2388.03,8133.23,8.4178,0.03,390,2388,100.0,38.95,23.3442
3,1,4,0.0007,0.0000,100.0,518.67,642.35,1582.79,1401.87,14.62,...,522.86,2388.08,8133.83,8.3682,0.03,392,2388,100.0,38.88,23.3739
4,1,5,-0.0019,-0.0002,100.0,518.67,642.37,1582.85,1406.22,14.62,...,522.19,2388.04,8133.80,8.4294,0.03,393,2388,100.0,38.90,23.4044



Testing Dataset
----------------------------------------


,id,cycle,setting1,setting2,setting3,s1,s2,s3,s4,s5,...,s12,s13,s14,s15,s16,s17,s18,s19,s20,s21
0,1,1,0.0023,0.0003,100.0,518.67,643.02,1585.29,1398.21,14.62,...,521.72,2388.03,8125.55,8.4052,0.03,392,2388,100.0,38.86,23.3735
1,1,2,-0.0027,-0.0003,100.0,518.67,641.71,1588.45,1395.42,14.62,...,522.16,2388.06,8139.62,8.3803,0.03,393,2388,100.0,39.02,23.3916
2,1,3,0.0003,0.0001,100.0,518.67,642.46,1586.94,1401.34,14.62,...,521.97,2388.03,8130.10,8.4441,0.03,393,2388,100.0,39.08,23.4166
3,1,4,0.0042,0.0000,100.0,518.67,642.44,1584.12,1406.42,14.62,...,521.38,2388.05,8132.90,8.3917,0.03,391,2388,100.0,39.00,23.3737
4,1,5,0.0014,0.0000,100.0,518.67,642.51,1587.19,1401.92,14.62,...,522.15,2388.03,8129.54,8.4031,0.03,390,2388,100.0,38.99,23.4130



Truth Dataset
----------------------------------------


,id,cycle
0,1,112
1,2,98
2,3,69
3,4,82
4,5,91


Data Quality Assessment

In [5]:
def data_quality_report(df, name):

    report = pd.DataFrame({
        "Data Type": df.dtypes,
        "Missing Values": df.isnull().sum(),
        "Missing (%)": (df.isnull().mean() * 100).round(2),
        "Unique Values": df.nunique(),
        "Constant Feature": df.nunique().le(1)
    })

    print(f"\n{name} Data Quality Report")
    print("=" * 50)

    print("Dataset Shape:", df.shape)
    print("Duplicate Rows:", df.duplicated().sum())

    display(report)

In [6]:
datasets = {
    "Training": train_df,
    "Testing": test_df,
    "Truth": truth_df
}

for name, df in datasets.items():
    data_quality_report(df, name)


Training Data Quality Report
Dataset Shape: (20631, 26)
Duplicate Rows: 0


,Data Type,Missing Values,Missing (%),Unique Values,Constant Feature
id,int64,0,0.0,100,False
cycle,int64,0,0.0,362,False
setting1,float64,0,0.0,158,False
setting2,float64,0,0.0,13,False
setting3,float64,0,0.0,1,True
s1,float64,0,0.0,1,True
s2,float64,0,0.0,310,False
s3,float64,0,0.0,3012,False
s4,float64,0,0.0,4051,False
s5,float64,0,0.0,1,True



Testing Data Quality Report
Dataset Shape: (13096, 26)
Duplicate Rows: 0


,Data Type,Missing Values,Missing (%),Unique Values,Constant Feature
id,int64,0,0.0,100,False
cycle,int64,0,0.0,303,False
setting1,float64,0,0.0,150,False
setting2,float64,0,0.0,14,False
setting3,float64,0,0.0,1,True
s1,float64,0,0.0,1,True
s2,float64,0,0.0,262,False
s3,float64,0,0.0,2361,False
s4,float64,0,0.0,2954,False
s5,float64,0,0.0,1,True



Truth Data Quality Report
Dataset Shape: (100, 2)
Duplicate Rows: 0


,Data Type,Missing Values,Missing (%),Unique Values,Constant Feature
id,int64,0,0.0,100,False
cycle,int64,0,0.0,71,False


Statistical Analysis

In [7]:
def statistical_summary(df, name):

    print(f"\n{name} Statistical Summary")
    print("=" * 50)

    display(df.describe().T.round(3))

for name, df in datasets.items():
    statistical_summary(df, name)


Training Statistical Summary


,count,mean,std,min,25%,50%,75%,max
id,20631.0,51.507,29.228,1.000,26.000,52.000,77.000,100.000
cycle,20631.0,108.808,68.881,1.000,52.000,104.000,156.000,362.000
setting1,20631.0,-0.000,0.002,-0.009,-0.002,0.000,0.002,0.009
setting2,20631.0,0.000,0.000,-0.001,-0.000,0.000,0.000,0.001
setting3,20631.0,100.000,0.000,100.000,100.000,100.000,100.000,100.000
s1,20631.0,518.670,0.000,518.670,518.670,518.670,518.670,518.670
s2,20631.0,642.681,0.500,641.210,642.325,642.640,643.000,644.530
s3,20631.0,1590.523,6.131,1571.040,1586.260,1590.100,1594.380,1616.910
s4,20631.0,1408.934,9.001,1382.250,1402.360,1408.040,1414.555,1441.490
s5,20631.0,14.620,0.000,14.620,14.620,14.620,14.620,14.620



Testing Statistical Summary


,count,mean,std,min,25%,50%,75%,max
id,13096.0,51.544,28.289,1.000,28.000,52.000,76.000,100.000
cycle,13096.0,76.837,53.058,1.000,33.000,69.000,113.000,303.000
setting1,13096.0,-0.000,0.002,-0.008,-0.002,-0.000,0.002,0.008
setting2,13096.0,0.000,0.000,-0.001,-0.000,-0.000,0.000,0.001
setting3,13096.0,100.000,0.000,100.000,100.000,100.000,100.000,100.000
s1,13096.0,518.670,0.000,518.670,518.670,518.670,518.670,518.670
s2,13096.0,642.475,0.401,641.130,642.198,642.460,642.740,644.300
s3,13096.0,1588.099,5.003,1569.040,1584.600,1587.990,1591.362,1607.550
s4,13096.0,1404.735,6.688,1384.390,1399.950,1404.440,1409.050,1433.360
s5,13096.0,14.620,0.000,14.620,14.620,14.620,14.620,14.620



Truth Statistical Summary


,count,mean,std,min,25%,50%,75%,max
id,100.0,50.50,29.011,1.0,25.75,50.5,75.25,100.0
cycle,100.0,75.52,41.765,7.0,32.75,86.0,112.25,145.0


# **Part 2 – Data Cleaning**

- Checked for missing values.
- Checked for duplicate records.
- Verified categorical features and encoding requirements.
- Identified and removed constant features.


Handle Missing Values

In [8]:
missing_values = train_df.isnull().sum().sum()

print("Total Missing Values:", missing_values)

Total Missing Values: 0


In [9]:
print("Testing Missing Values:",
      test_df.isnull().sum().sum())

Testing Missing Values:

 0


Handle Duplicate Records

In [10]:
print("Training Duplicates:", train_df.duplicated().sum())
print("Testing Duplicates:", test_df.duplicated().sum())

Training Duplicates: 0
Testing Duplicates: 0


Encoding Verification

In [11]:
categorical_features = train_df.select_dtypes(
    include=["object", "category"]
).columns.tolist()

print("Categorical Features:", categorical_features)

if not categorical_features:
    print("Encoding is not required. All features are numerical.")
else:
    print("Encoding is required.")

Categorical Features: []
Encoding is not required. All features are numerical.


# **Part 3 – Prepare the Target Variable**

- Calculated Remaining Useful Life (RUL) for the training data using engine cycles.

Calculate RUL
$$RUL = Maximum\ Cycle - Current\ Cycle$$

In [12]:
train_df = add_rul(train_df)

display(train_df[["id", "cycle", "RUL"]].head(10))

,id,cycle,RUL
0,1,1,191
1,1,2,190
2,1,3,189
3,1,4,188
4,1,5,187
5,1,6,186
6,1,7,185
7,1,8,184
8,1,9,183
9,1,10,182


# **Part 4 – Train / Validation Split**

- Split the training data into training and validation sets using GroupShuffleSplit.
- Ensured that engine IDs do not overlap between the two sets to prevent data leakage.

Separate Features and Target

In [13]:
X = train_df.drop(columns=["RUL", "id"])
y = train_df["RUL"]

groups = train_df["id"]

In [14]:
print("Train Data Columns:", train_df.shape[1])
print("Features Before Split:", X.shape[1])
print("Feature Names:", X.columns.tolist())

Train Data Columns: 27
Features Before Split: 25
Feature Names: ['cycle', 'setting1', 'setting2', 'setting3', 's1', 's2', 's3', 's4', 's5', 's6', 's7', 's8', 's9', 's10', 's11', 's12', 's13', 's14', 's15', 's16', 's17', 's18', 's19', 's20', 's21']


Split Data by Engine

In [15]:
train_idx, val_idx = split_by_engine(train_df, val_size=VAL_SIZE, seed=SEED)

X_train = X.iloc[train_idx]
X_val = X.iloc[val_idx]

y_train = y.iloc[train_idx]
y_val = y.iloc[val_idx]

print("Training Shape:", X_train.shape)
print("Validation Shape:", X_val.shape)

print("Training Engines:", groups.iloc[train_idx].nunique())
print("Validation Engines:", groups.iloc[val_idx].nunique())

Training Shape: (16561, 25)
Validation Shape: (4070, 25)
Training Engines: 80
Validation Engines: 20


Remove Constant Features

In [16]:
constant_features = find_constant_features(X_train)

print("Constant Features:")
print(constant_features)

X_train = X_train.drop(columns=constant_features)
X_val = X_val.drop(columns=constant_features)

print("Training Shape:", X_train.shape)
print("Validation Shape:", X_val.shape)

Constant Features:
['setting3', 's1', 's5', 's10', 's16', 's18', 's19']
Training Shape: (16561, 18)
Validation Shape: (4070, 18)


# **Part 5- Feature Scaling**

 - Applied StandardScaler.
 - Fitted the scaler on training data only.
 - Applied the same transformation to validation and testing data.

Apply StandardScaler

In [17]:
scaler = fit_scaler(X_train)

X_train_scaled = scaler.transform(X_train)
X_val_scaled = scaler.transform(X_val)

print("Training Data Scaled:", X_train_scaled.shape)
print("Validation Data Scaled:", X_val_scaled.shape)

Training Data Scaled: (16561, 18)
Validation Data Scaled: (4070, 18)


Scale Testing Data

In [18]:
X_test = test_df.drop(columns=["id"])

X_test = X_test.drop(columns=constant_features)

X_test_scaled = scaler.transform(X_test)

print("Testing Data Scaled:", X_test_scaled.shape)
print("Training Features:", X_train_scaled.shape[1])
print("Validation Features:", X_val_scaled.shape[1])
print("Testing Features:", X_test_scaled.shape[1])

Testing Data Scaled: (13096, 18)
Training Features: 18
Validation Features: 18
Testing Features: 18


In [19]:
print("Training and Validation features match:",
      list(X_train.columns) == list(X_val.columns))

print("Training and Testing features match:",
      list(X_train.columns) == list(X_test.columns))

Training and Validation features match: True
Training and Testing features match: True


# **Part 6 – Sequence Preparation**

 - Created time-series sequences using a window size of 30 cycles.
 - Prepared training, validation, and testing sequences.

Define Window Size

In [20]:
WINDOW_SIZE = CONFIG["window_size"]
print("Window size:", WINDOW_SIZE)

Window size: 30


Prepare Scaled Data

In [21]:
feature_cols = X_train.columns.tolist()

train_data = train_df.iloc[train_idx][["id", "cycle", "RUL"]].copy()
val_data = train_df.iloc[val_idx][["id", "cycle", "RUL"]].copy()

train_data[feature_cols] = X_train_scaled
val_data[feature_cols] = X_val_scaled

test_data = test_df[["id", "cycle"]].copy()
test_data[feature_cols] = X_test_scaled

print("Scaled data prepared successfully.")

Scaled data prepared successfully.


Create the Sequences (function in `src/data.py`)

Create Training and Validation Sequences

In [22]:
X_train_seq, y_train_seq = create_sequences(
    train_data, feature_cols, WINDOW_SIZE
)

X_val_seq, y_val_seq = create_sequences(
    val_data, feature_cols, WINDOW_SIZE
)

print("Training Sequences:", X_train_seq.shape)
print("Training Targets:", y_train_seq.shape)

print("Validation Sequences:", X_val_seq.shape)
print("Validation Targets:", y_val_seq.shape)

Training Sequences: (14241, 30, 18)
Training Targets: (14241,)
Validation Sequences: (3490, 30, 18)
Validation Targets: (3490,)


Create Testing Sequences

In [23]:
X_test_seq = create_test_sequences(
    test_data, feature_cols, WINDOW_SIZE
)

print("Testing Sequences:", X_test_seq.shape)

Testing Sequences: (100, 30, 18)


Final Shape Verification

In [24]:
print("Training Shape:", X_train_seq.shape)
print("Validation Shape:", X_val_seq.shape)
print("Testing Shape:", X_test_seq.shape)

Training Shape: (14241, 30, 18)
Validation Shape: (3490, 30, 18)
Testing Shape: (100, 30, 18)


# **Part 7 – Test Target Preparation (Actual RUL)**

- Retrieved actual RUL values from the ground truth dataset.
- Verified that test sequences and actual RUL values are correctly aligned.

Prepare Actual Test RUL

In [25]:
# Get the test engine IDs in sorted order
test_engine_ids = sorted(test_data["id"].unique())

# Get the actual RUL for each test engine
truth_lookup = truth_df.set_index("id")["cycle"]

# Match actual RUL values with test engine IDs
y_test = truth_lookup.reindex(test_engine_ids).to_numpy(dtype=float)

print("Actual Test RUL Shape:", y_test.shape)
print("First 10 Actual RUL Values:", y_test[:10])

Actual Test RUL Shape: (100,)
First 10 Actual RUL Values: [112.  98.  69.  82.  91.  93.  91.  95. 111.  96.]


Verify Test Targets

In [26]:
# Check that the number of targets matches the number of test sequences
assert len(y_test) == len(X_test_seq), \
    "Mismatch between test sequences and actual RUL values!"

# Check for missing target values
assert not np.isnan(y_test).any(), \
    "Missing RUL values detected!"

# Verify that all test engine IDs exist in the truth dataset
assert set(test_engine_ids) == set(truth_df["id"]), \
    "Mismatch between test engine IDs and truth IDs!"

print("Test Target Verification: Passed")
print("Number of Test Sequences:", len(X_test_seq))
print("Number of Actual RUL Values:", len(y_test))

Test Target Verification: Passed
Number of Test Sequences: 100
Number of Actual RUL Values: 100


# **Part 8 – Export the Prepared Data for the Team**

- Saves the prepared arrays to `prepared_data.npz` so the model members (baselines and Neural Networks) all start from exactly the same data.
- The file is **not** committed to GitHub (it is listed in `.gitignore`); everyone regenerates it by running this notebook.

In [27]:
save_prepared_data("prepared_data.npz",
    X_train_seq=X_train_seq, y_train_seq=y_train_seq,
    X_test_seq=X_test_seq,   y_test=y_test,
    val_id=val_data["id"].to_numpy(), val_cycle=val_data["cycle"].to_numpy(),
    val_rul=val_data["RUL"].to_numpy(), val_features=val_data[feature_cols].to_numpy(),
    feature_cols=np.array(feature_cols), window_size=np.array(WINDOW_SIZE))

print("Saved prepared_data.npz")

# Save the fitted scaler for later inference (predict.py).
scaler_path = os.path.join(REPO_ROOT, "models", "scaler.joblib")
save_scaler(scaler, scaler_path)
print("Saved", scaler_path)

Saved prepared_data.npz
Saved /tmp/rul/models/scaler.joblib
